# 04 — Prétraitement

Ce notebook explique et justifie les choix de prétraitement de House Prices. Les règles réutilisables sont importées depuis `src/preprocessing/`. Nous partons des anomalies et des distributions observées à l’[étape 3](03_analyse_exploratoire.ipynb).

## 1. Protocole

Dans les CSV, `NA` signale une valeur absente. Le mot `None` est un code valide, notamment pour une maison sans parement de maçonnerie. Nous désactivons donc les autres marqueurs de valeurs manquantes de pandas.

Nous définissons la partition 80/20 des expériences RAG et une validation croisée à cinq plis sur les 1 460 ventes étiquetées. Ce sont deux protocoles distincts. La comparaison ci-dessous utilise les prédictions hors pli de toutes les ventes. La partition 20 % recoupe ces plis et ne constitue donc pas un test indépendant de ce choix. Tout paramètre appris, dont les médianes et les coefficients du modèle, provient des seules maisons d’entraînement du pli. Le fichier Kaggle `test.csv` sert ici à compter et illustrer les corrections.

In [ ]:
from pathlib import Path
import inspect

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from src.evaluation import rmse_log
from src.preprocessing.steps.cleaning import Cleaner, LARGE_HOUSE_AREA, remove_outliers

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
read_options = {"na_values": ["NA"], "keep_default_na": False}
train = pd.read_csv(root / "data/train.csv", **read_options)
test = pd.read_csv(root / "data/test.csv", **read_options)
X, y = train.drop(columns="SalePrice"), train["SalePrice"]
train_index, holdout_index = train_test_split(X.index, test_size=0.2, random_state=42)
folds = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"Partition 80/20 : {len(train_index)} maisons d’entraînement et {len(holdout_index)} de validation.")

La partition réserve 292 ventes sur 1 460. En validation croisée, chacune des 1 460 maisons contribue une seule prédiction à la comparaison.

## 2. Nettoyage

Nous corrigeons les erreurs manifestes et marquons comme inconnue toute information contradictoire dont la valeur correcte nous échappe. L’imputation relève de la section 3. Une catégorie rare peut encore décrire des maisons différentes. La section 7 évaluera l’intérêt prédictif des variables conservées.

### Doublons et identifiant

Un identifiant unique pourrait masquer des doublons descriptifs. Nous vérifions séparément les identifiants, les lignes complètes et les caractéristiques seules.

In [ ]:
display(pd.Series({
    "Identifiants répétés": train["Id"].duplicated().sum(),
    "Lignes identiques": train.duplicated().sum(),
    "Caractéristiques identiques hors Id et prix": X.drop(columns="Id").duplicated().sum(),
}, name="Répétitions dans train"))
cleaner = Cleaner().fit(X.iloc[train_index])
raw = pd.concat({"train": X, "test": test}, names=["Fichier", "Ligne"])
clean = pd.concat({"train": cleaner.transform(X), "test": cleaner.transform(test)},
                  names=raw.index.names)

def rule_doc(column):
    """Afficher la ligne de documentation correspondant à la règle étudiée."""
    lines = inspect.getdoc(Cleaner.transform).splitlines()
    print("Cleaner.transform —", next(line for line in lines if line.startswith(column + " ")))

def changed(column):
    """Repérer une correction, en considérant deux valeurs manquantes comme identiques."""
    before, after = raw[column], clean[column]
    same = before.eq(after).fillna(False) | (before.isna() & after.isna())
    return ~same

Aucun identifiant ni aucune ligne complète ne se répète dans train. Nous gardons les ventes et retirons `Id` des prédicteurs. Nous le conservons dans les tableaux descriptifs pour retrouver chaque maison.

### 2.1 Type de logement

**Problème.** À l’étape 3 (§ 2), `MSSubClass` est lu comme un entier. Le tableau regroupe toutes les maisons concernées par code et par fichier.

**Décision.** Convertir le code en texte. **Code.** Règle `MSSubClass` de `Cleaner.transform`.

In [ ]:
rule_doc("MSSubClass")
display(pd.crosstab(raw["MSSubClass"], raw.index.get_level_values("Fichier")))

**Justification.** Le code 20 désigne une maison à un étage construite à partir de 1946, le code 60 une maison à deux étages de la même période, et le code 190 une conversion bifamiliale. Un coefficient appliqué au nombre imposerait un ordre et une distance arbitraires. L’encodage catégoriel pourra aussi traiter le code 150, observé seulement dans test.

### 2.2 Revêtements extérieurs

**Problème.** `Exterior1st` indique le revêtement principal des murs et `Exterior2nd` le revêtement secondaire. L’étape 3 (§ 2) relève trois différences d’orthographe entre ces colonnes.

**Décision.** Employer les codes du revêtement principal pour les deux colonnes. **Code.** Règle `Exterior2nd` de `Cleaner.transform`.

In [ ]:
rule_doc("Exterior2nd")
exterior = raw.loc[changed("Exterior2nd"), ["Id", "Exterior1st", "Exterior2nd"]].copy()
exterior["Code harmonisé"] = clean.loc[exterior.index, "Exterior2nd"]
display(exterior.groupby(["Exterior2nd", "Code harmonisé", "Fichier"]).size()
        .unstack("Fichier", fill_value=0))
display(exterior)

**Justification.** Une maison entièrement en bardeaux de bois pouvait porter à la fois `WdShing` et `Wd Shng`. Une comparaison directe aurait alors compté deux matériaux. La correction conserve l’information sur le matériau.

### 2.3 Services publics et variables quasi constantes

**Problème.** La synthèse de l’étape 3 (§ 8) signale sept variables quasi constantes. Nous comptons les modalités de `Utilities`, y compris les valeurs inconnues, puis examinons les exceptions des six autres variables.

**Décision.** Retirer `Utilities` et conserver les six autres caractéristiques. **Code.** Règle `Utilities` de `Cleaner.transform`.

In [ ]:
rule_doc("Utilities")
display(pd.crosstab(raw["Utilities"].fillna("Inconnu"), raw.index.get_level_values("Fichier")))
rare_columns = ["Street", "PoolQC", "Condition2", "RoofMatl", "Heating", "MiscFeature"]
rows = []
for column in rare_columns:
    values = train[column].fillna("Absent")
    majority = values.mode().iat[0]
    unusual = values.ne(majority)
    rows.append([column, majority, unusual.sum(), test[column].fillna("Absent").ne(majority).sum(),
                 y[~unusual].median(), y[unusual].median()])
display(pd.DataFrame(rows, columns=["Variable", "Majorité train", "Exceptions train",
                                     "Exceptions test", "Prix médian majorité ($)",
                                     "Prix médian exceptions ($)"]))

**Justification.** Une seule maison de train a la modalité `NoSeWa`. Dans test, toutes les modalités renseignées sont `AllPub`. Les autres variables ont des exceptions dans les deux fichiers et les prix médians diffèrent dans train. Les petits effectifs limitent l’interprétation de ces écarts. La section 7 examinera leur intérêt pour la prédiction.

### 2.4 Année du garage

**Problème.** Deux garages de test ont une année postérieure à la vente (étape 3, § 4). Le tableau rapproche l’année déclarée du type de garage et des dates de la maison.

**Décision.** Si l’année du garage dépasse celle de la vente de plus d’un an, reprendre l’année de rénovation de cette même maison. **Code.** Règle `GarageYrBlt` de `Cleaner.transform`.

In [ ]:
rule_doc("GarageYrBlt")
garage = raw.loc[raw["GarageYrBlt"].gt(raw["YrSold"]),
                 ["Id", "GarageType", "GarageYrBlt", "YearBuilt", "YearRemodAdd", "YrSold"]].copy()
garage["Année retenue"] = clean.loc[garage.index, "GarageYrBlt"]
display(garage)

**Justification.** Les ventes vont de 2006 à 2010. L’année 2207 du garage attaché de la maison 2593 est une faute de frappe manifeste. Sa rénovation date de 2007, année plausible que la règle retrouve sans consulter l’identifiant. Kaggle exige une prédiction pour chaque maison de test. Garder la maison, l’existence de son garage et une année approximative préserve l’information disponible. Pour la maison 2550, une vente sur plan en 2007 est compatible avec une construction en 2008. La tolérance d’un an conserve cette date.

### 2.5 Parement de maçonnerie

**Problème.** Le parement est une couche décorative de brique ou de pierre sur une partie des murs extérieurs. `MasVnrType` décrit le matériau et `MasVnrArea` la surface en pi². L’étape 3 (§ 4) signale des contradictions entre les deux. Pour situer leur ordre de grandeur, nous calculons la surface médiane des parements de surface positive et de matériau connu dans train.

**Décision.** Corriger la surface bouche-trou. Pour les autres contradictions, marquer le seul champ douteux comme inconnu. **Code.** Règle `MasVnrType` de `Cleaner.transform`.

In [ ]:
rule_doc("MasVnrType")
known = train["MasVnrType"].isin(["BrkFace", "BrkCmn", "Stone"]) & train["MasVnrArea"].gt(0)
print(f"Surface médiane des parements renseignés : {train.loc[known, 'MasVnrArea'].median():g} pi²")
area_changed = changed("MasVnrArea")
type_changed = changed("MasVnrType")
placeholder = area_changed & clean["MasVnrArea"].eq(0)
unknown_area = area_changed & clean["MasVnrArea"].isna()
masonry = raw.loc[type_changed | area_changed, ["Id", "MasVnrType", "MasVnrArea"]].copy()
masonry["Type retenu"] = clean.loc[masonry.index, "MasVnrType"]
masonry["Surface retenue"] = clean.loc[masonry.index, "MasVnrArea"]
display(masonry)

**Justification.** Un parement couvre typiquement environ 200 pi². Une surface d’un pied carré pour un parement déclaré absent est une valeur bouche-trou. À l’inverse, plusieurs centaines de pieds carrés rendent le matériau déclaré absent suspect. Quand le matériau est connu et que la surface vaut zéro, c’est la mesure de surface qui manque.

| Cas | Id dans train | Id dans test | Champ probablement faux et correction |
|---|---|---|---|
| Aucun parement, 1 pi² | 774, 1231 | 2453 | Surface bouche-trou ramenée à 0 |
| Aucun parement, surface supérieure à 1 pi² | 625, 1301, 1335 | 1670 | Type inconnu, surface conservée |
| Brique ou pierre, 0 pi² | 689, 1242 | 2320 | Surface inconnue, matériau conservé |

La section 3 devra distinguer ces inconnues créées par le nettoyage des `NA` d’origine lors de l’imputation.

### 2.6 Valeurs aberrantes

Une vente est aberrante lorsque son prix paraît incohérent avec les caractéristiques de la maison. Un prix élevé peut très bien correspondre à une vaste maison de grande qualité. Nous reprenons les quatre grandes maisons du nuage de points de l’étape 3 (§ 5 et § 7.1).

In [ ]:
display(train.loc[X["GrLivArea"].gt(LARGE_HOUSE_AREA),
                  ["Id", "Neighborhood", "GrLivArea", "OverallQual", "SaleCondition", "SalePrice"]])
print(inspect.getdoc(remove_outliers))

Les maisons 524 et 1299, de qualité 10 à Edwards, se sont vendues moins de 185 000 $ en condition `Partial`. Les deux grandes maisons de NoRidge se sont vendues beaucoup plus cher. Nous comparons le retrait ciblé au retrait par surface seule et au retrait par prix seul.

**Comparaison provisoire.** Un transformateur de pipeline modifie X et doit préserver l’alignement avec y. Nous filtrons donc ensemble X et les prix en dollars **avant** d’ajuster le pipeline, dans la partie entraînement de chaque pli. Toutes les ventes de validation sont prédites.

Les étapes 3 à 7 restent à développer. Pour cette comparaison, nous utilisons uniquement `Cleaner`, une imputation médiane des variables numériques, `log1p` des surfaces (zéro reste défini), une standardisation numérique et un encodage disjonctif des catégories après remplacement provisoire des inconnues par la constante `None`. Les catégories inédites sont ignorées. RidgeCV ajuste une régression ridge sur le logarithme naturel du prix avec 30 pénalisations candidates communes aux quatre options.

In [ ]:
# Les surfaces sont exprimées en pi² dans le dictionnaire d'Ames.
area_columns = [
    "LotArea", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "GarageArea", "WoodDeckSF",
    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea",
]

def other_numeric(frame):
    """Les autres grandeurs restent sur leur échelle avant standardisation."""
    return frame.select_dtypes(include="number").columns.difference(area_columns).tolist()

def provisional_model():
    """Construire un pipeline neuf pour que chaque ajustement reste propre au pli."""
    areas = make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log1p),
                          StandardScaler())
    numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    nominal = make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    columns = ColumnTransformer([
        ("areas", areas, area_columns), ("numeric", numeric, other_numeric),
        ("nominal", nominal, make_column_selector(dtype_exclude="number")),
    ])
    return make_pipeline(Cleaner(), columns, RidgeCV(alphas=np.logspace(-1, 3, 30)))

Le seuil du 1 % pour l’option D est recalculé sur les prix d’entraînement de chaque pli. Pour les scores, nous rassemblons les prédictions hors pli en dollars. Nous calculons ensuite la RMSE des logarithmes sur toutes les maisons, puis sur les 5 % les plus chères d’après les prix observés. Ce dernier seuil sert uniquement à lire les résultats.

In [ ]:
options = ["A · Tout garder", "B · Retrait ciblé", "C · Surface > 4 000 pi²", "D · Prix du centile supérieur"]
scores = []
for option in options:
    predictions = pd.Series(np.nan, index=y.index)
    removed = []
    for fit_index, valid_index in folds.split(X):
        features, prices = X.iloc[fit_index], y.iloc[fit_index]
        if option == options[1]:
            features, prices = remove_outliers(features, prices)
        elif option == options[2]:
            keep = features["GrLivArea"].le(LARGE_HOUSE_AREA)
            features, prices = features.loc[keep], prices.loc[keep]
        elif option == options[3]:
            keep = prices.le(prices.quantile(0.99))
            features, prices = features.loc[keep], prices.loc[keep]
        removed.append(len(fit_index) - len(features))
        model = provisional_model().fit(features, np.log(prices))
        predictions.iloc[valid_index] = np.exp(model.predict(X.iloc[valid_index]))
    assert predictions.notna().all()
    expensive = y.ge(y.quantile(0.95))
    scores.append([option, removed, rmse_log(y, predictions),
                   rmse_log(y[expensive], predictions[expensive])])

comparison = pd.DataFrame(scores, columns=["Option", "Retraits par pli", "RMSE log · toutes",
                                          "RMSE log · 5 % les plus chères"])
display(comparison.round(4))

**Interprétation.** Le retrait du centile supérieur (D) prive le modèle de ventes représentatives du haut de gamme. Sur les 5 % les plus chères, l’erreur passe de 0,1772 (A) à 0,1993 (D). Une maison de test dépasse également 4 000 pi², même si son prix est inconnu. B et C sont proches sur toutes les ventes (0,1283 et 0,1276). B donne la plus faible erreur sur les maisons chères (0,1448 contre 0,1474). Dans le troisième pli, les deux ventes ciblées se trouvent en validation et contribuent toutes deux au score.

**Décision.** Nous retenons B. De Cock conseille pour son jeu pédagogique d’écarter les maisons de plus de 4 000 pi². Il distingue les ventes partielles atypiques des très grandes maisons vendues à un prix cohérent. Sur le train Kaggle, B conserve les deux ventes de NoRidge qui suivent la tendance générale. La section 8 reprendra la comparaison avec la recette complète. [De Cock (2011), *Journal of Statistics Education*, 19(3)](https://doi.org/10.1080/10691898.2011.11889627)

### 2.7 Récapitulatif du nettoyage

Les suppressions de colonnes et la conversion de type concernent chaque ligne. Pour les corrections de valeurs, nous comptons les différences observées entre les données brutes et la sortie de `Cleaner`. La dernière ligne compte le retrait ciblé sur l’ensemble de train à titre descriptif.

In [ ]:
def counts(mask):
    """Compter les maisons concernées dans chacun des deux fichiers."""
    return [int(mask.loc[file].sum()) for file in ("train", "test")]

sizes = [len(train), len(test)]
retained, _ = remove_outliers(X, y)
summary = pd.DataFrame([
    ["Doublons", train.duplicated().sum(), test.duplicated().sum(), "Aucun retrait"],
    ["Id", *sizes, "Retirer la colonne"],
    ["MSSubClass", *sizes, "Convertir en texte"],
    ["Exterior2nd", *counts(changed("Exterior2nd")), "Harmoniser trois codes"],
    ["Utilities", *sizes, "Retirer la colonne"],
    ["GarageYrBlt", *counts(changed("GarageYrBlt")), "Reprendre YearRemodAdd"],
    ["Parement absent, surface minime", *counts(placeholder), "Ramener la surface à 0"],
    ["Parement absent, surface > 1", *counts(type_changed), "Marquer le type inconnu"],
    ["Matériau connu, surface nulle", *counts(unknown_area), "Marquer la surface inconnue"],
    ["Ventes aberrantes (B)", len(X) - len(retained), 0, "Retirer de l’entraînement seulement"],
], columns=["Règle", "Maisons dans train", "Maisons dans test", "Action"])
display(summary)

Le nettoyage conserve toutes les lignes nécessaires aux prédictions Kaggle. Les contradictions de maçonnerie laissent une information exploitable dans l’autre champ. La prochaine étape traitera les valeurs manquantes en tenant compte de cet indice.

## 3. Valeurs manquantes

À compléter.

## 4. Création de caractéristiques

À compléter.

## 5. Encodage des variables catégorielles

À compléter.

## 6. Normalisation et standardisation

À compléter.

## 7. Sélection des variables (stratégies)

À compléter.

## 8. Pipelines et reproductibilité

À compléter.

## 9. Bilan

À compléter.